# 环境安装

使用[langchain集成unstructured](https://docs.langchain.com/oss/python/integrations/document_loaders/unstructured_file#installation-for-local)的本地安装

```sh
# 基础依赖
# brew install libmagic poppler tesseract
# tesseract（OCR 引擎）
# tesseract：负责 OCR（把图片里的字认出来）。
# poppler：负责 解析和转换 PDF。Unstructured 需要它来提取 PDF 里的文字、把 PDF 页面转成图片（方便 OCR 识别）

# 官网的安装，langchain-unstructured[local]，其实这个 [local] 选项在底层默认引入的是 unstructured[all-docs]
# uv add "langchain-unstructured[local]"

# 自己的精简安装
uv add langchain-unstructured 

```

[unstructured](https://docs.unstructured.io/open-source/installation/full-installation)官方网站的的安装需要包

```sh
uv add "unstructured[md]"
```

# UnstructuredLoader 进行切分MD

[langchain UnstructuredLoader API](https://reference.langchain.com/python/langchain-unstructured/document_loaders/UnstructuredLoader)

In [18]:
from langchain_unstructured import UnstructuredLoader

# loader = UnstructuredLoader(
#     file_path="./assets/sample.md",
#     chunking_strategy="basic",
#     max_characters=1000000,
#     include_orig_elements=False,
#     strategy="fast",
# )

loader = UnstructuredLoader(
    file_path="./assets/sample.md",
    chunking_strategy="by_title",
    max_characters=500,
    include_orig_elements=False,
    strategy="fast",
)

In [19]:
documents = loader.load()

In [ ]:
print(type(documents), len(documents))
print(documents)

<class 'list'> 9
[Document(metadata={'source': './assets/sample.md', 'languages': ['nor', 'zho', 'cat'], 'file_directory': './assets', 'filename': 'sample.md', 'filetype': 'text/markdown', 'last_modified': '2026-08-29T00:13:11', 'category': 'CompositeElement', 'element_id': '4d9af05d38a2ed6be8cfa36ab446089f'}, page_content='投机解码（Speculative Decoding）介绍\n\n引言\n\n投机解码（Speculative Decoding）是一种先进的推理优化技术，用于加速大型语言模型（Large Language Models, LLMs）的自回归文本生成过程，同时确保输出质量不受影响。该技术灵感来源于计算机体系结构中的投机执行（Speculative Execution），旨在解决LLMs在生成长序列时因串行依赖导致的延迟问题。通过引入一个小型草稿模型（Draft Model）来“猜测”多个未来token，然后由目标大型模型（Target Model）并行验证这些猜测，投机解码可显著提升推理速度，通常达到2x至3x的加速倍数，而不改变模型的输出分布。'), Document(metadata={'source': './assets/sample.md', 'languages': ['nor', 'eng'], 'file_directory': './assets', 'filename': 'sample.md', 'filetype': 'text/markdown', 'last_modified': '2026-08-29T00:13:11', 'category': 'CompositeElement', 'element_id': 'b93b0e43948dd43a66c484232c63918e'}, page_content='历史与发展\n\n投机解码的概念最早可追溯到2022年的研究论文《Fast Infer

libmagic 是一个用来识别文件真实类型的底层库

时我们判断文件类型靠看后缀名（比如 .md、.pdf），但这很容易被伪造或改错。libmagic 的做法是直接读取文件开头的一小段二进制数据，比对内置的“魔数”数据库，从而准确判断出它到底是什么格式。

它其实就是 Unix/Linux 系统里 file 命令背后干活的那个核心引擎。

In [24]:
print(f"✅ 成功加载了 {len(documents)} 个 Document 对象\n")
for i, doc in enumerate(documents):
    print(f"--- 📄 Document {i + 1} ---")
    print(doc.page_content)
    print("=" * 50)

✅ 成功加载了 9 个 Document 对象

--- 📄 Document 1 ---
投机解码（Speculative Decoding）介绍

引言

投机解码（Speculative Decoding）是一种先进的推理优化技术，用于加速大型语言模型（Large Language Models, LLMs）的自回归文本生成过程，同时确保输出质量不受影响。该技术灵感来源于计算机体系结构中的投机执行（Speculative Execution），旨在解决LLMs在生成长序列时因串行依赖导致的延迟问题。通过引入一个小型草稿模型（Draft Model）来“猜测”多个未来token，然后由目标大型模型（Target Model）并行验证这些猜测，投机解码可显著提升推理速度，通常达到2x至3x的加速倍数，而不改变模型的输出分布。
--- 📄 Document 2 ---
历史与发展

投机解码的概念最早可追溯到2022年的研究论文《Fast Inference from Transformers via Speculative Decoding》（arXiv:2211.17192），由Google研究团队提出。该论文针对Transformer模型的推理瓶颈，引入了使用近似模型生成候选token并由目标模型并行验证的机制，在T5-XXL模型上实现了2x-3x加速。

2023年，另一篇关键论文《Accelerating Large Language Model Decoding with Speculative Sampling》（arXiv:2302.01318）扩展了这一想法，提出投机采样算法，在Chinchilla 70B模型上达到了2-2.5x的分布式加速。同年，变体如Medusa（arXiv:2401.10774）引入树状注意力机制和多解码头，进一步提高了效率。
--- 📄 Document 3 ---
到2024-2025年，投机解码的变体层出不穷，包括自投机解码（Self-Speculative Decoding）、EAGLE和多模态投机解码（Multimodal Speculative Decoding, MSD）。一项全面综述《Unlocking Efficiency in Large Language Model Inference: A Compr

## 参数小结


这套参数配置非常有代表性，其实是**为构建 RAG（检索增强生成）系统做准备的经典配置**

 📋 **参数总结表**

| 参数名                      | 你设置的值             | 作用与含义                                                   | 为什么要这么设？                                             |
| :-------------------------- | :--------------------- | :----------------------------------------------------------- | :----------------------------------------------------------- |
| **`file_path`**             | `"./assets/sample.md"` | **目标路径**。告诉加载器去哪里读取文件。                     | 指向你要解析的具体文件（这里是一篇长文）。                   |
| **`chunking_strategy`**     | `"by_title"`           | **切分策略**。基于文档的标题结构（如 Markdown 的 `#`、`##`）进行语义切分。 | 与之前的 `basic`（盲目堆叠到上限）不同，`by_title` 会**智能地保留章节的完整性**，不会把两个无关的小节强行揉在一起。 |
| **`max_characters`**        | `500`                  | **单块最大字符数**（硬性上限）。                             | 之前设 100 万是为了不切分，现在设 500 是为了**限制单块体积**。在 RAG 中，传给大模型的上下文（Context）不能太长，500 字符刚好适合作为检索的基本单元。 |
| **`include_orig_elements`** | `False`                | **是否保留原始元素元数据**。                                 | 设为 `False` 可以让返回的 `Document` 更加干净。如果设为 `True`，每个块里会塞满一大堆原始的坐标、字体大小等无用信息，徒增 Token 消耗。 |
| **`strategy`**              | `"fast"`               | **解析策略**。                                               | `"fast"` 使用基于规则的轻量级提取，速度极快。如果是扫描版 PDF 或复杂表格，才需要改成 `"hi_res"`（高精度，会调用深度学习模型，但速度慢）。 |

---

💡 深入理解：`by_title` 和 `max_characters` 是怎么配合的？

如果查看刚才打印出来的 `documents`，你会发现一个有趣的现象：

1. **优先按标题切**：如果你的某个小节（比如“引言”）只有 200 个字，它不会去凑字数，而是直接把这一整个小节作为一个 `Document` 返回。
2. **超长强制截断**：如果你的某个小节（比如“核心原理”）有 1500 个字，它会在塞满 500 个字后，强行把剩下 1000 个字切分成多个 `Document`。

这就是 `by_title` 比普通的文本分割器更聪明的地方——它**既尊重了文档原本的语义结构，又用 `max_characters` 兜底防止了文本超长**。


因为你 `max_characters=500`，它只是设定了一个**天花板**。

- 字数 >500 的段落：被按段落或句子强行截断成多个块。
- 字数<500 的段落：只要遇到了`下一个同级或更高级的标题`，就直接独立成块。

